In [3]:
library(dplyr)
library(tidyr)
library(pheatmap)
library(viridis)
library(tools)
library(cluster)
library(igraph)
library(tibble)

library(clusterProfiler)
library(org.Hs.eg.db)
library(dplyr)

In [4]:
# 读入
signature_NK <- read.csv(
  "/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.3.cNMF_NK/cnmf_NK_out_Sample_counts/NK_cGEP_Top30_Genes_spectra_normalized_fil.csv",row.names = 1,
  check.names = FALSE,
  stringsAsFactors = FALSE)
# 转成 list（核心）
cluster_genes <- lapply(signature_NK, function(x) {
  x[!is.na(x) & x != ""]
})


gene_df <- data.frame(
  cGEP_Cluster = names(cluster_genes),  # 确保这里是 cGEP1, cGEP2...
  cGEP_signature = sapply(cluster_genes, function(x) paste(x, collapse = ",")),
  stringsAsFactors = FALSE
)
gene_df

,cGEP_Cluster,cGEP_signature
,<chr>,<chr>
cGEP1,cGEP1,"CYTB,COX2,ATP6,ND4L,COX3,ND1,EIF1,ND5,6h.2,ATP8,H3F3B,AC107241.1,HLA-H,4h.2,C20orf24,PTMA,0h.1,FTH1,24h.2,ND4,LITAF,RBM39,SRSF5,ND2,DNAJB6,TMEM56,CEMIP2,RPS14P3,PDE4D,ND3"
cGEP2,cGEP2,"PFN1,ACTB,FCER1G,HSPA1A,IFITM1,TMSB10,MYL12A,NKG7,SERF2,TMSB4X,GSTP1,ND3,PPIA,ATP5E,IFITM2,ALOX5AP,AC245427.1,CFL1,TYROBP,ATP5F1E,HSPA1B,DNAJB1,MTRNR2L2,CLIC1,ATP5MF,MIF,HCST,B2M,RAC2,RPL39"
cGEP3,cGEP3,"RPS12,RPS2,RPS3,RPS27,RPS18,RPS15A,RPS14,EEF1A1,RPL13,RPL3,RPS4X,RPS28,RPL32,RP11-761N21.2,RP11-864N7.2,RPL34,RPL10,RPL23A,RPS23,RPL18A,RPS6,RPL19,RPS27A,RPL26,RPS8,PLAAT4,RPL30,RPS19,RPS29,RPL11"
cGEP4,cGEP4,"DNAJB1,HSPA1A,HSP90AA1,HSPE1,HSPA1B,HSPH1,HSPD1,BAG3,HSPA6,CACYBP,ZFAND2A,HSPA8,COL21A1,HSP90AB1,HSPB1,HSP90AA2P,DNAJA1,SERPINH1,PPP1R15A,UBC,NR4A1,ATF3,FKBP4,DNAJB4,MRPL18,JUN,DOK2,TFF1,TNFSF14,DEDD2"
cGEP5,cGEP5,"ZFP36,BTG2,NR4A2,JUNB,NR4A1,CD69,TNFAIP3,LOC284454,CXCR4,FOSB,RP11-463O12.5,NFKBIA,MCL1,JUN,PPP1R15A,AREG,IER2,DUSP2,NFKBIZ,ZNF331,FOS,AC027290.2,KDM6B,TSC22D3,REL,CSRNP1,DNAJA1,DUSP1,EGR1,NR4A3"
cGEP6,cGEP6,"IGFBP1,ITIH3,AL158850.1,MT-ND4,MT-CO3,MT-CO2,MT-CYB,MT-ATP6,MT-ND1,MT-ND3,MT-ND2,LBP,MT-CO1,RHEX,PNCK,MT-ND5,FGL1,FGA,ORM1,IGHV4-34,MT-ND4L,APOB,AL445673.1,NTM,CRP,PLA2G2A,SAA1,ROR1,UGT2B7,F2"
cGEP7,cGEP7,"BTG1,ITM2C,RPL13A,KRT81,RPS20,CAPG,RPS26,GZMK,RPL27A,KRT86,MALAT1,RPS2,CXCR6,CD44,RPS29,RPL37A,TXNIP,CXCR4,LINC00996,CD9,APOBEC3G,PNRC1,SLFN5,CXCR3,VIM,EPHA1,AC026366.1,RPLP2,RPL10A,SRGAP3"
cGEP8,cGEP8,"RPS12,RPL13,RPLP1,TPT1,IL7R,CCR6,RPL32,NCR2,RPL30,EEF1A1,RPL10,RPS23,APOC4-APOC2,AL445686.2,RPL28,GPR183,RPL34,RPS8,RPL8,RPLP0,VIM,AMICA1,RPS18,U62317.3,RPS14,RPL11,RPL41,RPS5,RPS13,RPS25"
cGEP9,cGEP9,"NKG7,PRF1,SPON2,FGFBP2,GZMB,FCGR3A,SCGB1D2,B2M,ITGB2,CST7,PFN1,GZMA,ACTB,IGFBP7,GPR56,CX3CR1,CYBA,CFL1,AKR1C3,TMSB10,FCER1G,CD247,MT-COX1,PRSS23,MYL12A,LOC101060789,PLAC8,S100A4,EMP3,GZMH"


In [6]:
#NK_gep_anno <- read.csv('./3.3.NK_cGEP_ALL_ANNO/3.3.NK_cGEP_Anno_final.csv')
NK_gep_anno <- read.csv('/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/NK_cGEP41.csv')

NK_gep_anno

cGEP_Cluster,cGEP_Anno_Name,Category
<chr>,<chr>,<chr>
cGEP1,Low_Quality_Mito,Artifact
cGEP10,NK_Cycling_G2M,Functional
cGEP11,Doublet_CD8_T,Doublet
cGEP12,Doublet_Club_Cell,Doublet
cGEP13,Doublet_Mast,Doublet
cGEP14,NK_ISG_Response,Functional
cGEP15,Inflammatory_Chemokine,Functional
cGEP16,Doublet_Macrophage,Doublet
cGEP17,Doublet_Treg,Doublet


In [7]:
final_combined_df <- NK_gep_anno %>%
  left_join(gene_df, by = "cGEP_Cluster")
final_combined_df

cGEP_Cluster,cGEP_Anno_Name,Category,cGEP_signature
<chr>,<chr>,<chr>,<chr>
cGEP1,Low_Quality_Mito,Artifact,"CYTB,COX2,ATP6,ND4L,COX3,ND1,EIF1,ND5,6h.2,ATP8,H3F3B,AC107241.1,HLA-H,4h.2,C20orf24,PTMA,0h.1,FTH1,24h.2,ND4,LITAF,RBM39,SRSF5,ND2,DNAJB6,TMEM56,CEMIP2,RPS14P3,PDE4D,ND3"
cGEP10,NK_Cycling_G2M,Functional,"TYMS,STMN1,TUBA1B,UBE2C,SPC25,MKI67,AURKB,TK1,DLGAP5,ASPM,TUBB,CDC20,CDK1,NUSAP1,TOP2A,PCLAF,ZWINT,CDC45,CCNB2,HMGB2,CENPF,UHRF1,CKAP2L,KIF15,BIRC5,KIF23,CENPW,CENPA,KIF2C,CDCA5"
cGEP11,Doublet_CD8_T,Doublet,"CD3D,CD3G,SIT1,CD8B,S100A6,TRGV11,LAG3,CD3E,S100A4,CD8A,CXCL13,TRGV2,CD5,TRBV14,TRAV8-1,TRBV6-1,TRBV11-3,TRAV38-1,TRBV27,SH3BGRL3,CD52,TMEM155,TRAC,TRAV9-2,VCAM1,TRAV8-2,CD27,LINC02446,THEMIS,F5"
cGEP12,Doublet_Club_Cell,Doublet,"SCGB3A2,RP5-851M4.1,FABP4,PGC,RPS29,GS1-600G8.5,SLC4A10,RPL13A,RPS20,GS1-259H13.2,CUEDC1,RPLP2,LTB,RP11-1143G9.4,RNASE2,RP11-589M4.1,SCGB1A1,RPL27A,RPS17,PIFO,IL7R,RPL41,TREM1,TRAC,ALOX15B,RPL31,RPS18,RPL23A,RPS2,AL121845.1"
cGEP13,Doublet_Mast,Doublet,"MS4A2,HDC,TPSAB1,TPSB2,SLC18A2,HPGDS,MAOB,KIT,CTTNBP2,CPA3,SDPR,CCL2,CPM,MITF,ALDH1A1,RGS13,DHRS9,SPDYE1,ALOX5,LPCAT2,TMEM255B,GATA2,IL1RL1,VWA5A,DPP6,CRISPLD2,HPGD,PROS1,CLEC3B,CNRIP1"
cGEP14,NK_ISG_Response,Functional,"ISG15,IFIT3,IFI6,MX1,RSAD2,USP18,IFIT1,CMPK2,EPSTI1,OAS1,MX2,ISG20,IFI44L,IFI35,PLSCR1,LY6E,IRF7,IFITM1,TNFSF10,HERC5,EIF2AK2,LAMP3,IFI16,AC090377.1,ERICH3,IFIT2,OASL,OAS3,BST2,XAF1"
cGEP15,Inflammatory_Chemokine,Functional,"AC026801.2,CXCL2,AC124242.1,AQP9,CCL20,CLEC4E,AIF1,LILRA5,FCN1,CST3,BASP1,LINC01272,CCDC149,C5AR1,HBEGF,IL1B,LST1,HCK,ANKRD30B,MS4A7,SMIM25,CSF1R,TNFAIP6,CLEC7A,ENHO,BLOC1S1-RDH5,HLA-DRA,LOC100130520,FTL,RAB31"
cGEP16,Doublet_Macrophage,Doublet,"CST3,C1QC,C1QB,HLA-DRA,C1QA,APOE,CD74,HLA-DRB1,HLA-DPA1,SELENOP,HLA-DRB5,HLA-DPB1,AIF1,HLA-DQA1,RNASE1,APOC1,HLA-DQB1,MS4A6A,CD68,LINC01736,FTL,CD209,FCGRT,CSF1R,LGMN,MS4A7,TREM2,CD1E,CTSB,CD5L"
cGEP17,Doublet_Treg,Doublet,"LINC01943,AC017002.3,IL2RA,CTLA4,TRAC,TNFRSF4,IL1R2,FOXP3,ICOS,CD4,ZBTB32,AC133644.2,IL32,BATF,TNFRSF9,CD27,F5,TBC1D4,CAVIN3,GADD45A,CD28,CD3D,TNFRSF18,PKM,DNPH1,RTKN2,ID3,KCNN4,DUSP4,SLAMF1"


In [9]:
#OUT_FILE <- "./3.3.NK_cGEP_ALL_ANNO/3.3.NK_cGEP_Anno_Complete_With_Genes.csv"
OUT_FILE <- '/sibcb1/bioinformatics/yangyue/project/immunotherapy/7.6.cNMF_menchmark/cGEP_signature/NK_cGEP41_signature.csv'

write.csv(final_combined_df, OUT_FILE, row.names = FALSE)